# NLP - Task 2: LLMs for AI Feedback

## Project Objective
Optimize LLM prompts to improve consistency with human preferences.

## Experimental Methods
1. Baseline Prompt
2. Few-shot Prompting
3. Chain-of-Thought Prompting
4. Self-Consistency
5. Multi-Agent Evaluation

## Evaluation Metrics
- Human Preference Consistency Rate
- Position Bias
- Response Latency
- Token Usage

In [5]:
%pip -q install langchain-core langchain-deepseek pandas tqdm

In [6]:
import os
import json
import re
import time
import random
import copy

import pandas as pd
from tqdm.auto import tqdm

from google.colab import userdata

from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Environment initialized successfully!")

Environment initialized successfully!


In [7]:
# Load API key securely from Colab Secrets

api_key = userdata.get("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("Please configure DEEPSEEK_API_KEY in Colab Secrets.")

os.environ["DEEPSEEK_API_KEY"] = api_key
os.environ["DEEPSEEK_BASE_URL"] = "https://api.deepseek.com/v1"

print("API configuration completed!")

API configuration completed!


In [8]:
llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0
)

parser = StrOutputParser()

test_chain = llm | parser

response = test_chain.invoke(
    "Reply with exactly: Connection successful."
)

print(response)

Connection successful.


In [9]:
!wget -q -O 1.rlhf.jsonl https://NLP-course-cuhksz.github.io/Assignments/Assignment1/task2/data/1.rlhf.jsonl

In [10]:
with open("1.rlhf.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print("Total samples:", len(data))

print("\nDataset fields:")
print(data[0].keys())

print("\nFirst sample:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Total samples: 100

Dataset fields:
dict_keys(['Question', 'Answer1', 'Answer2', 'Preference'])

First sample:
{
  "Question": "Human: What are some aspects of religious Islam that are compatible with western civilization?",
  "Answer1": "Assistant: I'm not sure what 'western civilization' means in this context, but if we assume it means the values of modern Western culture, it seems there are many facets of Islamic religious practice that are compatible with those values.  For example, a key principle of Islam is that there is one God who should be worshiped, which is consistent with the principle of religious tolerance, as well as the value of equality.  There are different branches of Islam, but many of them require",
  "Answer2": "Assistant: Well in a basic sense it is possible for Muslims to live in the Western world and take part in Western civilization. In fact, many Muslims have been at the center of important cultural developments of the last centuries.",
  "Preference": "Answ

In [11]:
from collections import Counter

# Check preference distribution
preference_counts = Counter(
    item["Preference"] for item in data
)

print("Preference distribution:")
for label, count in preference_counts.items():
    print(f"{label}: {count}")

# Check dataset completeness
required_fields = {
    "Question", "Answer1", "Answer2", "Preference"
}

print("\nDataset completeness:")
print("Total samples:", len(data))
print(
    "Complete samples:",
    sum(required_fields.issubset(item.keys()) for item in data)
)

# Display three examples
print("\nSample examples:")
for i, item in enumerate(data[:3]):
    print(f"\nExample {i+1}")
    print("Question:", item["Question"])
    print("Preference:", item["Preference"])

Preference distribution:
Answer2: 45
Answer1: 55

Dataset completeness:
Total samples: 100
Complete samples: 100

Sample examples:

Example 1
Question: Human: What are some aspects of religious Islam that are compatible with western civilization?
Preference: Answer2

Example 2
Question: Human: I want to learn how to make a New York style cheesecake.  Can you help me?
Preference: Answer1

Example 3
Question: Human: I know that for employment, many  companies want a potential employee to take a drug test. With THC in particular, how sensitive or specific can that type of drug test be? I know that with other drugs like alcohol and cocaine, they can pinpoint use within one to two days. Is there a way to test specifically for when someone last used marijuana?
Preference: Answer1


# 1. Baseline Prompt
## 1.1 Define the Baseline Prompt

In [12]:
def baseline_prompt(item):

    prompt = f"""
You are an impartial AI response evaluator.

Your task is to compare two answers to the same question
and predict which answer a human would prefer.

Evaluate the answers based on:
1. Relevance to the question
2. Factual correctness
3. Completeness
4. Clarity
5. Helpfulness

Do not automatically prefer a longer answer.
Judge both answers based on their overall quality.

Question:
{item['Question']}

Answer1:
{item['Answer1']}

Answer2:
{item['Answer2']}

Which answer would a human prefer?

Reply with exactly one label:
Answer1 or Answer2.
"""

    return prompt

## 1.2 Test on One Sample

In [13]:
import time
import re

# Select the first sample
sample = data[0]

# Generate prompt
prompt = baseline_prompt(sample)

# Call DeepSeek
start_time = time.perf_counter()

response = llm.invoke(prompt)

end_time = time.perf_counter()

# Extract prediction
raw_output = response.content

match = re.search(
    r"\bAnswer\s*([12])\b",
    raw_output,
    re.IGNORECASE
)

prediction = (
    "Answer" + match.group(1)
    if match else "Unparseable"
)

# Display results
print("Question:", sample["Question"])
print("Human Preference:", sample["Preference"])
print("Model Prediction:", prediction)
print("Raw Output:", raw_output)
print("Latency:", round(end_time - start_time, 3), "seconds")
print("Correct:", prediction == sample["Preference"])

Question: Human: What are some aspects of religious Islam that are compatible with western civilization?
Human Preference: Answer2
Model Prediction: Answer2
Raw Output: Answer2
Latency: 0.884 seconds
Correct: True


## 1.3 Run Baseline on the Full Dataset


In [14]:
import time
import re
import pandas as pd

# Store baseline experiment results
baseline_results = []


# Extract token usage from API response
def get_token_usage(response):

    usage = getattr(response, "usage_metadata", None)

    if not usage:
        metadata = getattr(response, "response_metadata", {}) or {}
        usage = (
            metadata.get("token_usage")
            or metadata.get("usage")
            or {}
        )

    if hasattr(usage, "model_dump"):
        usage = usage.model_dump()

    if not isinstance(usage, dict):
        usage = {}

    input_tokens = usage.get(
        "input_tokens",
        usage.get("prompt_tokens")
    )

    output_tokens = usage.get(
        "output_tokens",
        usage.get("completion_tokens")
    )

    total_tokens = usage.get("total_tokens")

    if total_tokens is None:
        if input_tokens is not None and output_tokens is not None:
            total_tokens = input_tokens + output_tokens

    return input_tokens, output_tokens, total_tokens


# Run baseline evaluation on all samples
for i, item in enumerate(data):

    start_time = time.perf_counter()

    try:
        # Generate prompt
        prompt = baseline_prompt(item)

        # Call DeepSeek
        response = llm.invoke(prompt)

        end_time = time.perf_counter()

        # Extract model output
        raw_output = response.content

        if isinstance(raw_output, list):
            raw_output = " ".join(
                block.get("text", "")
                if isinstance(block, dict)
                else str(block)
                for block in raw_output
            )

        raw_output = str(raw_output)

        # Extract predicted preference
        matches = re.findall(
            r"\bAnswer\s*([12])\b",
            raw_output,
            re.IGNORECASE
        )

        prediction = (
            "Answer" + matches[-1]
            if matches
            else "Unparseable"
        )

        # Extract token usage
        input_tokens, output_tokens, total_tokens = (
            get_token_usage(response)
        )

        error_message = None

    except Exception as e:

        end_time = time.perf_counter()

        prediction = "Error"
        raw_output = str(e)

        input_tokens = None
        output_tokens = None
        total_tokens = None

        error_message = str(e)

    # Calculate latency
    latency = end_time - start_time

    # Compare with human preference
    human_preference = item["Preference"]

    correct = prediction == human_preference

    # Store results
    baseline_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "correct": correct,
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    # Display progress
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(data)} samples")


# Convert results into DataFrame
baseline_df = pd.DataFrame(baseline_results)

print("\nBaseline evaluation completed!")
print("Total samples:", len(baseline_df))

Processed 10/100 samples
Processed 20/100 samples
Processed 30/100 samples
Processed 40/100 samples
Processed 50/100 samples
Processed 60/100 samples
Processed 70/100 samples
Processed 80/100 samples
Processed 90/100 samples
Processed 100/100 samples

Baseline evaluation completed!
Total samples: 100


## 1.4 Evaluate Baseline Results

In [15]:
# Calculate evaluation metrics

total_samples = len(baseline_df)

correct_predictions = baseline_df["correct"].sum()

consistency_rate = (
    correct_predictions / total_samples * 100
)

valid_predictions = baseline_df["prediction"].isin(
    ["Answer1", "Answer2"]
)

parse_rate = valid_predictions.mean() * 100

successful_latency = baseline_df.loc[
    valid_predictions, "latency_seconds"
]

average_latency = successful_latency.mean()


# Token usage statistics
input_usage = baseline_df["input_tokens"].dropna()
output_usage = baseline_df["output_tokens"].dropna()
total_usage = baseline_df["total_tokens"].dropna()


print("===== Baseline Evaluation Results =====")

print(f"Total samples: {total_samples}")

print(
    f"Correct predictions: {correct_predictions}"
)

print(
    f"Human Preference Consistency Rate: "
    f"{consistency_rate:.2f}%"
)

print(
    f"Valid Prediction Rate: {parse_rate:.2f}%"
)

print(
    f"Average Response Latency: "
    f"{average_latency:.3f} seconds"
    if not successful_latency.empty
    else "Average Response Latency: N/A"
)

if not total_usage.empty:
    print(f"Total Input Tokens: {input_usage.sum()}")
    print(f"Total Output Tokens: {output_usage.sum()}")
    print(f"Total Tokens: {total_usage.sum()}")
else:
    print("Token usage metadata: Not available")


# Save results
baseline_df.to_csv(
    "baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to baseline_results.csv")

===== Baseline Evaluation Results =====
Total samples: 100
Correct predictions: 64
Human Preference Consistency Rate: 64.00%
Valid Prediction Rate: 100.00%
Average Response Latency: 0.964 seconds
Total Input Tokens: 24955
Total Output Tokens: 200
Total Tokens: 25155

Results saved to baseline_results.csv


### 1.4.1 Inspect Prediction Results

In [16]:
display(
    baseline_df[
        [
            "sample_id",
            "human_preference",
            "prediction",
            "correct",
            "latency_seconds"
        ]
    ].head(15)
)

,sample_id,human_preference,prediction,correct,latency_seconds
0,1,Answer2,Answer2,True,0.790816
1,2,Answer1,Answer1,True,0.733928
2,3,Answer1,Answer2,False,1.015345
3,4,Answer1,Answer1,True,1.314962
4,5,Answer2,Answer2,True,0.574180
5,6,Answer2,Answer2,True,1.212414
6,7,Answer2,Answer1,False,1.088871
7,8,Answer1,Answer2,False,1.184936
8,9,Answer1,Answer2,False,1.063826
9,10,Answer2,Answer1,False,1.067966


## 2. Few-shot Prompting
### 2.1 Prepare Few-shot Examples

In [17]:
rng = random.Random(42)

few_shot_indices = []

for label in ["Answer1", "Answer2"]:
    candidates = [
        i for i, item in enumerate(data)
        if item["Preference"] == label
    ]

    few_shot_indices.extend(rng.sample(candidates, 2))

rng.shuffle(few_shot_indices)

few_shot_index_set = set(few_shot_indices)

few_shot_examples = [
    data[i] for i in few_shot_indices
]

print("Selected few-shot examples:")

for i, example in enumerate(few_shot_examples, 1):
    print(f"\nExample {i}")
    print("Human Preference:", example["Preference"])
    print("Question:", example["Question"][:150])

print("\nTotal demonstrations:", len(few_shot_examples))

Selected few-shot examples:

Example 1
Human Preference: Answer2
Question: Human: How long do blue whales nurture their young?

Example 2
Human Preference: Answer2
Question: Human: Can you list some Caribbean countries that use French as their official language?

Example 3
Human Preference: Answer1
Question: Human: Why do spanish people speak such a weird language?

Example 4
Human Preference: Answer1
Question: Human: What is Valhalla?

Total demonstrations: 4


### 2.2 Build Few-shot Prompt

In [18]:
def format_pair(item):
    return (
        f"Question: {item['Question']}\n"
        f"Answer1: {item['Answer1']}\n"
        f"Answer2: {item['Answer2']}"
    )


def build_few_shot_prompt(item, examples):

    demonstrations = []

    for i, example in enumerate(examples, 1):
        demonstrations.append(
            f"Example {i}\n"
            f"{format_pair(example)}\n"
            f"Human preference: {example['Preference']}"
        )

    examples_text = "\n\n".join(demonstrations)

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Consider helpfulness, relevance, clarity, correctness and safety.

Here are some examples:

{examples_text}

Now evaluate the following question:

{format_pair(item)}

Return only Answer1 or Answer2.
"""

    return prompt

### 2.3 Run Few-shot Evaluation

In [19]:
few_shot_results = []

for i, item in enumerate(data):

    if i in few_shot_index_set:
        continue

    start_time = time.perf_counter()

    try:
        prompt = build_few_shot_prompt(
            item, few_shot_examples
        )

        response = llm.invoke(prompt)

        raw_output = response.content

        if isinstance(raw_output, list):
            raw_output = " ".join(
                block.get("text", "")
                if isinstance(block, dict)
                else str(block)
                for block in raw_output
            )

        raw_output = str(raw_output)

        matches = re.findall(
            r"\bAnswer\s*([12])\b",
            raw_output,
            re.IGNORECASE
        )

        prediction = (
            "Answer" + matches[-1]
            if matches
            else "Unparseable"
        )

        input_tokens, output_tokens, total_tokens = (
            get_token_usage(response)
        )

        error_message = None

    except Exception as e:

        prediction = "Error"
        raw_output = str(e)

        input_tokens = None
        output_tokens = None
        total_tokens = None

        error_message = str(e)

    latency = time.perf_counter() - start_time

    few_shot_results.append({
        "sample_id": i + 1,
        "human_preference": item["Preference"],
        "prediction": prediction,
        "correct": prediction == item["Preference"],
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    if len(few_shot_results) % 10 == 0:
        print(
            f"Processed {len(few_shot_results)}/"
            f"{len(data) - len(few_shot_examples)} samples"
        )

few_shot_df = pd.DataFrame(few_shot_results)

print("\nFew-shot evaluation completed!")
print("Total test samples:", len(few_shot_df))

Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Few-shot evaluation completed!
Total test samples: 96


### 2.4 Evaluate Few-shot Results

In [20]:
total_samples = len(few_shot_df)

consistency_rate = (
    few_shot_df["correct"].mean() * 100
)

valid_rate = few_shot_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = few_shot_df.loc[
    few_shot_df["prediction"] != "Error",
    "latency_seconds"
].mean()

total_input_tokens = few_shot_df["input_tokens"].sum()
total_output_tokens = few_shot_df["output_tokens"].sum()
total_tokens = few_shot_df["total_tokens"].sum()

print("===== Few-shot Evaluation Results =====")

print("Total samples:", total_samples)

print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

few_shot_df.to_csv(
    "few_shot_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to few_shot_results.csv")

===== Few-shot Evaluation Results =====
Total samples: 96
Consistency Rate: 68.75%
Valid Prediction Rate: 100.00%
Average Latency: 0.966 seconds
Total Input Tokens: 86095
Total Output Tokens: 192
Total Tokens: 86287

Results saved to few_shot_results.csv


### 2.5 Compare with Baseline

In [21]:
baseline_test_df = baseline_df[
    baseline_df["sample_id"].isin(
        few_shot_df["sample_id"]
    )
]

comparison_df = pd.DataFrame({
    "Method": ["Baseline", "Few-shot"],
    "Consistency Rate": [
        baseline_test_df["correct"].mean() * 100,
        few_shot_df["correct"].mean() * 100
    ],
    "Average Latency": [
        baseline_test_df["latency_seconds"].mean(),
        few_shot_df.loc[
            few_shot_df["prediction"] != "Error",
            "latency_seconds"
        ].mean()
    ],
    "Total Tokens": [
        baseline_test_df["total_tokens"].sum(),
        few_shot_df["total_tokens"].sum()
    ]
})

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.971061,24079
1,Few-shot,68.750000,0.965903,86287


## 3. Chain-of-Thought Prompting
### 3.1 Build CoT Prompt

In [22]:
def build_cot_prompt(item):

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Evaluate the responses step by step:

1. Identify the main requirements of the question.
2. Compare the helpfulness and relevance of both responses.
3. Consider their correctness, clarity and safety.
4. Decide which response better satisfies the user's needs.

Provide a brief comparison and explain your decision.

Question: {item['Question']}

Answer1: {item['Answer1']}

Answer2: {item['Answer2']}

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt

### 3.2 Run CoT Evaluation

In [23]:
cot_results = []

for i, item in enumerate(data):

    if i in few_shot_index_set:
        continue

    start_time = time.perf_counter()

    try:
        prompt = build_cot_prompt(item)
        response = llm.invoke(prompt)

        raw_output = response.content

        if isinstance(raw_output, list):
            raw_output = " ".join(
                block.get("text", "")
                if isinstance(block, dict)
                else str(block)
                for block in raw_output
            )

        raw_output = str(raw_output)

        matches = re.findall(
            r"\bAnswer\s*([12])\b",
            raw_output,
            re.IGNORECASE
        )

        prediction = (
            "Answer" + matches[-1]
            if matches
            else "Unparseable"
        )

        input_tokens, output_tokens, total_tokens = (
            get_token_usage(response)
        )

        error_message = None

    except Exception as e:

        prediction = "Error"
        raw_output = str(e)

        input_tokens = None
        output_tokens = None
        total_tokens = None

        error_message = str(e)

    latency = time.perf_counter() - start_time

    cot_results.append({
        "sample_id": i + 1,
        "human_preference": item["Preference"],
        "prediction": prediction,
        "correct": prediction == item["Preference"],
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    if len(cot_results) % 10 == 0:
        print(
            f"Processed {len(cot_results)}/96 samples"
        )

cot_df = pd.DataFrame(cot_results)

print("\nCoT evaluation completed!")
print("Total test samples:", len(cot_df))

Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

CoT evaluation completed!
Total test samples: 96


### 3.3 Evaluate CoT Results

In [24]:
total_samples = len(cot_df)

consistency_rate = cot_df["correct"].mean() * 100

valid_rate = cot_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = cot_df.loc[
    cot_df["prediction"] != "Error",
    "latency_seconds"
].mean()

total_input_tokens = cot_df["input_tokens"].sum()
total_output_tokens = cot_df["output_tokens"].sum()
total_tokens = cot_df["total_tokens"].sum()

print("===== CoT Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

cot_df.to_csv(
    "cot_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to cot_results.csv")

===== CoT Evaluation Results =====
Total samples: 96
Consistency Rate: 68.75%
Valid Prediction Rate: 100.00%
Average Latency: 2.724 seconds
Total Input Tokens: 24943
Total Output Tokens: 28564
Total Tokens: 53507

Results saved to cot_results.csv


### 3.4 Compare Prompting Methods

In [25]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.971061,24079
1,Few-shot,68.750000,0.965903,86287
2,Chain-of-Thought,68.750000,2.723579,53507


## 4. Self-Consistency

### 4.1 Configure Repeated Sampling

In [26]:
sc_llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.7
)

n_votes = 5

print("Self-Consistency configured successfully!")
print("Number of reasoning samples:", n_votes)

Self-Consistency configured successfully!
Number of reasoning samples: 5


### 4.2 Build Self-Consistency Prompt

In [27]:
def build_sc_prompt(item):

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Consider the following criteria:
1. Helpfulness and relevance.
2. Correctness and accuracy.
3. Clarity and completeness.
4. Safety and appropriateness.

Evaluate the two responses carefully and make an independent decision.

Provide a brief explanation of your comparison.

Question: {item['Question']}

Answer1: {item['Answer1']}

Answer2: {item['Answer2']}

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt

### 4.3 Run Self-Consistency Evaluation

In [28]:
sc_results = []

for i, item in enumerate(data):

    if i in few_shot_index_set:
        continue

    start_time = time.perf_counter()

    votes = []
    raw_outputs = []

    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0

    error_messages = []

    for _ in range(n_votes):

        try:
            prompt = build_sc_prompt(item)
            response = sc_llm.invoke(prompt)

            raw_output = response.content

            if isinstance(raw_output, list):
                raw_output = " ".join(
                    block.get("text", "")
                    if isinstance(block, dict)
                    else str(block)
                    for block in raw_output
                )

            raw_output = str(raw_output)

            matches = re.findall(
                r"\bAnswer\s*([12])\b",
                raw_output,
                re.IGNORECASE
            )

            vote = (
                "Answer" + matches[-1]
                if matches
                else "Unparseable"
            )

            try:
                input_t, output_t, total_t = get_token_usage(response)

                total_input_tokens += input_t or 0
                total_output_tokens += output_t or 0
                total_tokens += total_t or 0

            except Exception:
                pass

        except Exception as e:

            vote = "Error"
            raw_output = str(e)
            error_messages.append(str(e))

        votes.append(vote)
        raw_outputs.append(raw_output)

    valid_votes = [
        v for v in votes
        if v in ["Answer1", "Answer2"]
    ]

    count_1 = valid_votes.count("Answer1")
    count_2 = valid_votes.count("Answer2")

    if count_1 > count_2:
        prediction = "Answer1"
    elif count_2 > count_1:
        prediction = "Answer2"
    else:
        prediction = "Unparseable"

    latency = time.perf_counter() - start_time

    sc_results.append({
        "sample_id": i + 1,
        "human_preference": item["Preference"],
        "prediction": prediction,
        "correct": prediction == item["Preference"],
        "votes": " | ".join(votes),
        "vote_1": votes[0],
        "vote_2": votes[1],
        "vote_3": votes[2],
        "vote_4": votes[3],
        "vote_5": votes[4],
        "latency_seconds": latency,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "raw_outputs": "\n---\n".join(raw_outputs),
        "error": "; ".join(error_messages) if error_messages else None
    })

    if len(sc_results) % 10 == 0:
        print(
            f"Processed {len(sc_results)}/96 samples"
        )

sc_df = pd.DataFrame(sc_results)

print("\nSelf-Consistency evaluation completed!")
print("Total test samples:", len(sc_df))

Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Self-Consistency evaluation completed!
Total test samples: 96


### 4.4 Evaluate Self-Consistency Results

In [29]:
total_samples = len(sc_df)

consistency_rate = sc_df["correct"].mean() * 100

valid_rate = sc_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = sc_df["latency_seconds"].mean()

total_input_tokens = sc_df["input_tokens"].sum()
total_output_tokens = sc_df["output_tokens"].sum()
total_tokens = sc_df["total_tokens"].sum()

print("===== Self-Consistency Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

sc_df.to_csv(
    "self_consistency_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to self_consistency_results.csv")

===== Self-Consistency Evaluation Results =====
Total samples: 96
Consistency Rate: 64.58%
Valid Prediction Rate: 100.00%
Average Latency: 8.867 seconds
Total Input Tokens: 120395
Total Output Tokens: 59658
Total Tokens: 180053

Results saved to self_consistency_results.csv


### 4.5 Compare Prompting Methods

In [30]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    },
    {
        "Method": "Self-Consistency",
        "Consistency Rate": sc_df["correct"].mean() * 100,
        "Average Latency": sc_df["latency_seconds"].mean(),
        "Total Tokens": sc_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.971061,24079
1,Few-shot,68.750000,0.965903,86287
2,Chain-of-Thought,68.750000,2.723579,53507
3,Self-Consistency,64.583333,8.866785,180053


## 5. Multi-Agent Evaluation

### 5.1 Configure Multi-Agent System

In [31]:
agent_llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0
)

agent_roles = [
    (
        "Accuracy Evaluator",
        "Focus on factual correctness, logical consistency, "
        "and the accuracy of the information."
    ),
    (
        "Helpfulness Evaluator",
        "Focus on relevance, usefulness, and how well "
        "the response addresses the user's needs."
    ),
    (
        "Quality and Safety Evaluator",
        "Focus on clarity, completeness, safety, and "
        "overall response quality."
    )
]

print("Multi-Agent system initialized!")
print("Number of agents:", len(agent_roles))

Multi-Agent system initialized!
Number of agents: 3


### 5.2 Build Agent Prompts

In [32]:
def build_agent_prompt(item, agent_name, focus):

    prompt = f"""
You are an independent AI evaluator named {agent_name}.

Your task is to compare two AI responses and predict
which one is more likely to be preferred by human annotators.

Your evaluation focus:
{focus}

Consider the following criteria:
1. Helpfulness and relevance.
2. Correctness and accuracy.
3. Clarity and completeness.
4. Safety and appropriateness.

Give more attention to your assigned evaluation focus.
Evaluate both answers fairly and independently.

Question:
{item['Question']}

Answer1:
{item['Answer1']}

Answer2:
{item['Answer2']}

Briefly explain your decision.

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt


def extract_agent_choice(raw_output):

    match = re.search(
        r"Final choice\s*:\s*(Answer[12])\b",
        raw_output,
        re.IGNORECASE
    )

    if match:
        return match.group(1).title()

    return "Unparseable"


print("Agent prompts and output parser ready!")

Agent prompts and output parser ready!


### 5.3 Run Multi-Agent Evaluation

In [33]:
multi_agent_results = []

for i, item in enumerate(data):

    if i in few_shot_index_set:
        continue

    start_time = time.perf_counter()

    agent_votes = []
    raw_outputs = []

    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0

    error_messages = []

    for agent_name, focus in agent_roles:

        try:
            prompt = build_agent_prompt(
                item,
                agent_name,
                focus
            )

            response = agent_llm.invoke(prompt)

            raw_output = response.content

            if isinstance(raw_output, list):
                raw_output = " ".join(
                    block.get("text", "")
                    if isinstance(block, dict)
                    else str(block)
                    for block in raw_output
                )

            raw_output = str(raw_output)

            vote = extract_agent_choice(raw_output)

            try:
                input_t, output_t, total_t = get_token_usage(response)

                total_input_tokens += input_t or 0
                total_output_tokens += output_t or 0
                total_tokens += total_t or 0

            except Exception:
                pass

        except Exception as e:

            vote = "Error"
            raw_output = str(e)
            error_messages.append(str(e))

        agent_votes.append(vote)
        raw_outputs.append(raw_output)

    valid_votes = [
        v for v in agent_votes
        if v in ["Answer1", "Answer2"]
    ]

    count_1 = valid_votes.count("Answer1")
    count_2 = valid_votes.count("Answer2")

    if count_1 >= 2 and count_1 > count_2:
        prediction = "Answer1"
    elif count_2 >= 2 and count_2 > count_1:
        prediction = "Answer2"
    else:
        prediction = "Unparseable"

    latency = time.perf_counter() - start_time

    multi_agent_results.append({
        "sample_id": i + 1,
        "human_preference": item["Preference"],
        "prediction": prediction,
        "correct": prediction == item["Preference"],
        "agent_1": agent_votes[0],
        "agent_2": agent_votes[1],
        "agent_3": agent_votes[2],
        "votes": " | ".join(agent_votes),
        "latency_seconds": latency,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "raw_outputs": "\n---\n".join(raw_outputs),
        "error": "; ".join(error_messages) if error_messages else None
    })

    if len(multi_agent_results) % 10 == 0:
        print(
            f"Processed {len(multi_agent_results)}/96 samples"
        )

multi_agent_df = pd.DataFrame(multi_agent_results)

print("\nMulti-Agent evaluation completed!")
print("Total test samples:", len(multi_agent_df))

Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Multi-Agent evaluation completed!
Total test samples: 96


### 5.4 Evaluate Multi-Agent Results

In [34]:
total_samples = len(multi_agent_df)

consistency_rate = multi_agent_df["correct"].mean() * 100

valid_rate = multi_agent_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = multi_agent_df["latency_seconds"].mean()

total_input_tokens = multi_agent_df["input_tokens"].sum()
total_output_tokens = multi_agent_df["output_tokens"].sum()
total_tokens = multi_agent_df["total_tokens"].sum()

print("===== Multi-Agent Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

multi_agent_df.to_csv(
    "multi_agent_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to multi_agent_results.csv")

===== Multi-Agent Evaluation Results =====
Total samples: 96
Consistency Rate: 66.67%
Valid Prediction Rate: 100.00%
Average Latency: 5.440 seconds
Total Input Tokens: 80877
Total Output Tokens: 38462
Total Tokens: 119339

Results saved to multi_agent_results.csv


### 5.5 Compare All Prompting Methods

In [35]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    },
    {
        "Method": "Self-Consistency",
        "Consistency Rate": sc_df["correct"].mean() * 100,
        "Average Latency": sc_df["latency_seconds"].mean(),
        "Total Tokens": sc_df["total_tokens"].sum()
    },
    {
        "Method": "Multi-Agent",
        "Consistency Rate": multi_agent_df["correct"].mean() * 100,
        "Average Latency": multi_agent_df["latency_seconds"].mean(),
        "Total Tokens": multi_agent_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.971061,24079
1,Few-shot,68.750000,0.965903,86287
2,Chain-of-Thought,68.750000,2.723579,53507
3,Self-Consistency,64.583333,8.866785,180053
4,Multi-Agent,66.666667,5.439643,119339


## 6. Additional Analysis

### 6.1 Paired Error Analysis

In [36]:
from scipy.stats import binomtest

cot_compare = cot_df[
    ["sample_id", "correct"]
].rename(columns={"correct": "cot_correct"})

multi_compare = multi_agent_df[
    ["sample_id", "correct"]
].rename(columns={"correct": "multi_agent_correct"})

paired_df = cot_compare.merge(
    multi_compare,
    on="sample_id",
    validate="one_to_one"
)

multi_only = (
    (paired_df["multi_agent_correct"]) &
    (~paired_df["cot_correct"])
).sum()

cot_only = (
    (paired_df["cot_correct"]) &
    (~paired_df["multi_agent_correct"])
).sum()

both_correct = (
    paired_df["cot_correct"] &
    paired_df["multi_agent_correct"]
).sum()

both_wrong = (
    (~paired_df["cot_correct"]) &
    (~paired_df["multi_agent_correct"])
).sum()

n_discordant = int(multi_only + cot_only)

p_value = (
    binomtest(
        int(multi_only),
        n=n_discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue
    if n_discordant > 0 else 1.0
)

print("===== Paired Error Analysis =====")

print("Total matched samples:", len(paired_df))
print("Both correct:", both_correct)
print("Both wrong:", both_wrong)
print("Multi-Agent only correct:", multi_only)
print("CoT only correct:", cot_only)

print(f"\nMcNemar exact test p-value: {p_value:.4f}")

print("\nCases improved by Multi-Agent:")
print(
    paired_df.loc[
        paired_df["multi_agent_correct"] &
        (~paired_df["cot_correct"]),
        "sample_id"
    ].tolist()
)

print("\nCases improved by CoT:")
print(
    paired_df.loc[
        paired_df["cot_correct"] &
        (~paired_df["multi_agent_correct"]),
        "sample_id"
    ].tolist()
)

===== Paired Error Analysis =====
Total matched samples: 96
Both correct: 60
Both wrong: 26
Multi-Agent only correct: 4
CoT only correct: 6

McNemar exact test p-value: 0.7539

Cases improved by Multi-Agent:
[9, 36, 83, 97]

Cases improved by CoT:
[10, 21, 62, 81, 84, 95]


### 6.2 Statistical Significance Testing

In [38]:

import numpy as np
import pandas as pd
import itertools

from scipy.stats import (
    chi2,
    binomtest,
    friedmanchisquare,
    wilcoxon
)

# ==========================================
# 6.2.1 Prepare Matched Experimental Data
# ==========================================

baseline_test_df = baseline_df[
    baseline_df["sample_id"].isin(
        few_shot_df["sample_id"]
    )
].copy()

method_dfs = {
    "Baseline": baseline_test_df,
    "Few-shot": few_shot_df,
    "Chain-of-Thought": cot_df,
    "Self-Consistency": sc_df,
    "Multi-Agent": multi_agent_df
}

required_columns = [
    "sample_id",
    "correct",
    "latency_seconds",
    "total_tokens"
]

reference_ids = baseline_test_df["sample_id"].tolist()

if len(reference_ids) != len(set(reference_ids)):
    raise ValueError("Duplicate sample IDs in Baseline.")

for method, df in method_dfs.items():

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{method} missing columns: {missing}"
        )

    if df["sample_id"].duplicated().any():
        raise ValueError(
            f"Duplicate sample IDs in {method}."
        )

    if set(df["sample_id"]) != set(reference_ids):
        raise ValueError(
            f"Sample IDs do not match in {method}."
        )

aligned = {
    method: df.set_index("sample_id").loc[reference_ids]
    for method, df in method_dfs.items()
}

accuracy_matrix = pd.DataFrame({
    method: aligned[method]["correct"].astype(int)
    for method in method_dfs
})

latency_matrix = pd.DataFrame({
    method: pd.to_numeric(
        aligned[method]["latency_seconds"],
        errors="coerce"
    )
    for method in method_dfs
})

tokens_matrix = pd.DataFrame({
    method: pd.to_numeric(
        aligned[method]["total_tokens"],
        errors="coerce"
    )
    for method in method_dfs
})

if not accuracy_matrix.isin([0, 1]).all().all():
    raise ValueError("Correctness must contain only 0 or 1.")

for name, matrix in [
    ("Latency", latency_matrix),
    ("Tokens", tokens_matrix)
]:
    if matrix.isna().any().any():
        raise ValueError(f"Invalid values in {name}.")

n = len(reference_ids)
k = len(method_dfs)

print("Matched experimental data prepared.")
print("Methods:", k)
print("Matched samples:", n)


# ==========================================
# 6.2.2 Overall Statistical Tests
# ==========================================

accuracy_values = accuracy_matrix.to_numpy()

row_sums = accuracy_values.sum(axis=1)
col_sums = accuracy_values.sum(axis=0)

total = accuracy_values.sum()

denominator = (
    k * total - np.square(row_sums).sum()
)

numerator = (
    (k - 1) *
    (k * np.square(col_sums).sum() - total ** 2)
)

if denominator > 0:
    cochran_q = numerator / denominator
    accuracy_p = chi2.sf(cochran_q, k - 1)
else:
    cochran_q = np.nan
    accuracy_p = np.nan

# Effect Size for Accuracy
accuracy_w = (
    cochran_q / (n * (k - 1))
    if np.isfinite(cochran_q) else np.nan
)

# Friedman Test: Latency
latency_result = friedmanchisquare(
    *[
        latency_matrix[method].to_numpy()
        for method in method_dfs
    ]
)

# Friedman Test: Total Tokens
tokens_result = friedmanchisquare(
    *[
        tokens_matrix[method].to_numpy()
        for method in method_dfs
    ]
)

latency_w = (
    latency_result.statistic / (n * (k - 1))
)

tokens_w = (
    tokens_result.statistic / (n * (k - 1))
)

overall_tests = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "Test": "Cochran's Q",
        "Statistic": cochran_q,
        "p-value": accuracy_p,
        "Effect Size": accuracy_w
    },
    {
        "Metric": "Latency",
        "Test": "Friedman Test",
        "Statistic": latency_result.statistic,
        "p-value": latency_result.pvalue,
        "Effect Size": latency_w
    },
    {
        "Metric": "Total Tokens",
        "Test": "Friedman Test",
        "Statistic": tokens_result.statistic,
        "p-value": tokens_result.pvalue,
        "Effect Size": tokens_w
    }
])

print("\n===== Overall Statistical Tests =====")
display(overall_tests.round(6))


# ==========================================
# 6.2.3 Pairwise Statistical Comparisons
# ==========================================

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (m - np.arange(m)) * sorted_p
    )

    adjusted_sorted = np.minimum(
        adjusted_sorted, 1.0
    )

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    return adjusted


def paired_bootstrap_ci(
    differences,
    n_boot=10000,
    seed=42
):

    differences = np.asarray(
        differences, dtype=float
    )

    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        len(differences),
        size=(n_boot, len(differences))
    )

    bootstrap_means = differences[indices].mean(axis=1)

    return np.quantile(
        bootstrap_means,
        [0.025, 0.975]
    )


pairwise_rows = []

methods = list(method_dfs.keys())

comparison_matrices = {
    "Accuracy": accuracy_matrix,
    "Latency": latency_matrix,
    "Total Tokens": tokens_matrix
}

for metric, matrix in comparison_matrices.items():

    for method_a, method_b in itertools.combinations(
        methods, 2
    ):

        a = matrix[method_a].to_numpy()
        b = matrix[method_b].to_numpy()

        differences = a - b

        if metric == "Accuracy":

            a_only = int(
                ((a == 1) & (b == 0)).sum()
            )

            b_only = int(
                ((a == 0) & (b == 1)).sum()
            )

            discordant = a_only + b_only

            if discordant == 0:
                p_value = 1.0
            else:
                p_value = binomtest(
                    a_only,
                    n=discordant,
                    p=0.5,
                    alternative="two-sided"
                ).pvalue

            scale = 100

        else:

            if np.allclose(differences, 0):
                p_value = 1.0
            else:
                p_value = wilcoxon(
                    a,
                    b,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue

            scale = 1

        ci_lower, ci_upper = paired_bootstrap_ci(
            differences
        )

        pairwise_rows.append({
            "Metric": metric,
            "Method A": method_a,
            "Method B": method_b,
            "Mean Difference (A-B)": (
                differences.mean() * scale
            ),
            "95% CI Lower": ci_lower * scale,
            "95% CI Upper": ci_upper * scale,
            "p-value": p_value
        })


pairwise_results = pd.DataFrame(pairwise_rows)


# ==========================================
# 6.2.4 Multiple Comparison Correction
# ==========================================

pairwise_results["Holm-adjusted p"] = np.nan

for metric in pairwise_results["Metric"].unique():

    mask = pairwise_results["Metric"] == metric

    pairwise_results.loc[
        mask, "Holm-adjusted p"
    ] = holm_adjust(
        pairwise_results.loc[
            mask, "p-value"
        ].to_numpy()
    )

pairwise_results["Significant"] = (
    pairwise_results["Holm-adjusted p"] < 0.05
)

print("\n===== Pairwise Statistical Comparisons =====")
display(pairwise_results.round(4))


# ==========================================
# 6.2.5 Export Statistical Results
# ==========================================

overall_tests.to_csv(
    "overall_statistical_tests.csv",
    index=False,
    encoding="utf-8-sig"
)

pairwise_results.to_csv(
    "pairwise_statistical_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nStatistical analysis completed.")

print("Saved files:")
print("1. overall_statistical_tests.csv")
print("2. pairwise_statistical_results.csv")


Matched experimental data prepared.
Methods: 5
Matched samples: 96

===== Overall Statistical Tests =====


,Metric,Test,Statistic,p-value,Effect Size
0,Accuracy,Cochran's Q,3.781818,0.436338,0.009848
1,Latency,Friedman Test,364.808333,0.000000,0.950022
2,Total Tokens,Friedman Test,383.208333,0.000000,0.997938



===== Pairwise Statistical Comparisons =====


,Metric,Method A,Method B,Mean Difference (A-B),95% CI Lower,95% CI Upper,p-value,Holm-adjusted p,Significant
0,Accuracy,Baseline,Few-shot,-5.2083,-12.5000,2.0833,0.2668,1.0000,False
1,Accuracy,Baseline,Chain-of-Thought,-5.2083,-12.5000,1.0417,0.2266,1.0000,False
2,Accuracy,Baseline,Self-Consistency,-1.0417,-6.2500,4.1667,1.0000,1.0000,False
3,Accuracy,Baseline,Multi-Agent,-3.1250,-10.4167,3.1250,0.5488,1.0000,False
4,Accuracy,Few-shot,Chain-of-Thought,0.0000,-8.3333,8.3333,1.0000,1.0000,False
5,Accuracy,Few-shot,Self-Consistency,4.1667,-3.1250,11.4583,0.3877,1.0000,False
6,Accuracy,Few-shot,Multi-Agent,2.0833,-5.2083,9.3750,0.7744,1.0000,False
7,Accuracy,Chain-of-Thought,Self-Consistency,4.1667,-2.0833,10.4167,0.3438,1.0000,False
8,Accuracy,Chain-of-Thought,Multi-Agent,2.0833,-4.1667,8.3333,0.7539,1.0000,False
9,Accuracy,Self-Consistency,Multi-Agent,-2.0833,-8.3333,3.1250,0.7266,1.0000,False



Statistical analysis completed.
Saved files:
1. overall_statistical_tests.csv
2. pairwise_statistical_results.csv


# 6.3 Efficiency and Pareto Analysis

In [39]:
efficiency_rows = []

for method in method_dfs:

    correct_count = accuracy_matrix[method].sum()
    total_tokens = tokens_matrix[method].sum()

    efficiency_rows.append({
        "Method": method,
        "Accuracy (%)": accuracy_matrix[method].mean() * 100,
        "Average Latency (s)": latency_matrix[method].mean(),
        "Average Tokens": tokens_matrix[method].mean(),
        "Total Tokens": total_tokens,
        "Correct per 1000 Tokens": (
            correct_count / total_tokens * 1000
            if total_tokens > 0 else np.nan
        )
    })

efficiency_df = pd.DataFrame(efficiency_rows)

# Identify Pareto-optimal methods
pareto_flags = []

for i, current in efficiency_df.iterrows():

    dominated = False

    for j, other in efficiency_df.iterrows():

        if i == j:
            continue

        no_worse = (
            other["Accuracy (%)"] >= current["Accuracy (%)"]
            and other["Average Latency (s)"] <= current["Average Latency (s)"]
            and other["Total Tokens"] <= current["Total Tokens"]
        )

        strictly_better = (
            other["Accuracy (%)"] > current["Accuracy (%)"]
            or other["Average Latency (s)"] < current["Average Latency (s)"]
            or other["Total Tokens"] < current["Total Tokens"]
        )

        if no_worse and strictly_better:
            dominated = True
            break

    pareto_flags.append(not dominated)

efficiency_df["Pareto Optimal"] = pareto_flags

efficiency_df = efficiency_df.sort_values(
    "Accuracy (%)",
    ascending=False
).reset_index(drop=True)

print("===== Efficiency and Pareto Analysis =====")
display(efficiency_df.round(4))

efficiency_df.to_csv(
    "efficiency_pareto_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: efficiency_pareto_analysis.csv")


===== Efficiency and Pareto Analysis =====


,Method,Accuracy (%),Average Latency (s),Average Tokens,Total Tokens,Correct per 1000 Tokens,Pareto Optimal
0,Few-shot,68.7500,0.9659,898.8229,86287,0.7649,True
1,Chain-of-Thought,68.7500,2.7236,557.3646,53507,1.2335,True
2,Multi-Agent,66.6667,5.4396,1243.1146,119339,0.5363,False
3,Self-Consistency,64.5833,8.8668,1875.5521,180053,0.3443,False
4,Baseline,63.5417,0.9711,250.8229,24079,2.5333,True



Saved: efficiency_pareto_analysis.csv
